# Lab 2: API Client Integration

In this lab you will build a production-ready LLM API client using **LiteLLM** and **OpenRouter**.

**By the end of this lab you will know how to:**
- Securely load API keys from environment variables
- Make your first LLM API call via LiteLLM
- Use LiteLLM's built-in retry logic
- Use LiteLLM's built-in response caching

## Step 1 — Environment Setup

We never hardcode API keys in source code. Instead, we load them from a `.env` file at runtime.

Create a `.env` file in this directory with:
```
OPENROUTER_API_KEY=sk-or-...
```

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # Reads .env from the current directory


def get_api_key() -> str:
    """Retrieve and validate the OpenRouter API key."""
    token = os.getenv("OPENROUTER_API_KEY")
    if not token:
        raise EnvironmentError(
            "OPENROUTER_API_KEY not found. "
            "Create a .env file with your key or set the environment variable."
        )
    return token


get_api_key()  # Validate early — fail fast if the key is missing
print("API key loaded successfully.")

API key loaded successfully.


## Step 2 — Your First API Call

[LiteLLM](https://docs.litellm.ai/docs/) provides a unified `completion()` interface across 100+ LLM providers.
We prefix the model name with `openrouter/` so LiteLLM knows to route through OpenRouter.

Key features:

- Direct Python library integration in your codebase
- Router with retry/fallback logic across multiple deployments (e.g. Azure/OpenAI) - Router
- Application-level load balancing and cost tracking
- Exception handling with OpenAI-compatible errors
- Observability callbacks (Lunary, MLflow, Langfuse, etc.)


In [18]:
from litellm import completion

MODEL_ID = "openrouter/nvidia/nemotron-3-super-120b-a12b:free"
messages = []

def chat(user_message):
    messages.append({"role": "user", "content": user_message})
    response = completion(model=MODEL_ID, messages=messages, max_tokens=150)
    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

print(chat("Explain vector databases in one paragraph."))
print(chat("Can you explain in simpler terms?"))
print(chat("Give me a real-world example."))


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

A vector database is a specialized data store designed to efficiently index, query, and retrieve high‑dimensional vectors—often embeddings generated by machine‑learning models—using similarity search techniques such as approximate nearest‑neighbor (ANN) algorithms, allowing fast retrieval of items that are semantically or structurally close to a query vector while supporting typical database features like scaling, persistence, and metadata filtering.
Provider List: https://docs.litellm.ai/docs/providers



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

Okay, the user asked for a simpler explanation of vector databases after my initial technical definition. Hmm, they probably found the first response too jargon-heavy—words like "approximate nearest-neighbor" or "high-dimensional embeddings" might have confused the

## Step 3 — Retry Logic

Free-tier APIs are rate-limited. Instead of writing manual retry loops, LiteLLM has a built-in `num_retries` parameter that automatically retries on `RateLimitError` and network errors with exponential backoff.

In [ ]:
response = completion(
    model=MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
                
    max_tokens=150,
    temperature=0.7,
    num_retries=3,   # LiteLLM retries automatically on rate limit / network errors
    timeout=120,
)

print(response.choices[0].message.content)

## Step 4 — Response Caching

During development, you often run the same queries many times. Caching avoids redundant API calls, saving both time and quota.

LiteLLM ships with built-in caching — one line to enable it globally.

In [17]:
import litellm

# Enable in-memory caching for all subsequent completion() calls
litellm.enable_cache(type="local")

model = MODEL_ID
messages = [{"role": "user", "content": "What is retrieval-augmented generation?"}]

print("--- First call (Cache MISS — hits API) ---")
result1 = completion(model=model, messages=messages, num_retries=3, timeout=120, max_tokens=200)
print(result1.choices[0].message.content[:200])

print("\n--- Second call (Cache HIT — served from memory) ---")
result2 = completion(model=model, messages=messages, num_retries=3, timeout=120, max_tokens=200)
print(result2.choices[0].message.content[:200])

print("\nResponses identical:", result1.choices[0].message.content == result2.choices[0].message.content)

--- First call (Cache MISS — hits API) ---

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Okay, the user is asking about retrieval-augmented generation (RAG). Let me start by recalling what I know about RAG. It's a technique that combines information retrieval with generative models to imp

--- Second call (Cache HIT — served from memory) ---
Provider List: https://docs.litellm.ai/docs/providers

Okay, the user is asking about retrieval-augmented generation (RAG). Let me start by recalling what I know about RAG. It's a technique that combines information retrieval with generative models to imp

Responses identical: True

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

